In [ ]:
import os

dataset_path = 'D:\python\project\PythonProject\ml diabetes project\deeplearnig/CNN/archive/brain-tumor-mri-dataset'
glioma = os.listdir(f'{dataset_path}/glioma')
meningioma = os.listdir(f"{dataset_path}/meningioma")
notumor = os.listdir(f"{dataset_path}/notumor")
pituitary = os.listdir(f"{dataset_path}/pituitary")

print(f"glioma : {len(glioma)}")
print(f"meningioma : {len(meningioma)}")
print(f"notumor : {len(notumor)}")
print(f"pituitary : {len(pituitary)}")


In [ ]:
import numpy as np
from PIL import Image

X =[]
y = []

for img_name in os.listdir(f"{dataset_path}/glioma"):
    img_path = f"{dataset_path}/glioma/{img_name}"
    img = Image.open(img_path).convert('RGB')
    img = img.resize((96,96))
    img_array = np.array(img ,dtype=np.float32) / 255.0
    X.append(img_array)
    y.append(0)

for img_name in os.listdir(f"{dataset_path}/meningioma"):
    img_path = f"{dataset_path}/meningioma/{img_name}"
    img = Image.open(img_path).convert("RGB")
    img = img.resize((96,96))
    img_array = np.array(img, dtype=np.float32) / 255.0
    X.append(img_array)
    y.append(1)

for img_name in os.listdir(f"{dataset_path}/notumor"):
    img_path = f"{dataset_path}/notumor/{img_name}"
    img = Image.open(img_path).convert('RGB')
    img = img.resize((96,96))
    img_array = np.array(img, dtype=np.float32) / 255.0
    X.append(img_array)
    y.append(2)

for img_name in os.listdir(f"{dataset_path}/pituitary"):
    img_path = f"{dataset_path}/pituitary/{img_name}"
    img = Image.open(img_path).convert('RGB')
    img = img.resize((96,96))
    img_array = np.array(img ,dtype=np.float32) / 255.0
    X.append(img_array)
    y.append(3)

X = np.array(X, dtype=np.float32)
y = np.array(y)

print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split


X_train , X_test ,y_train ,y_test = train_test_split(X ,y , test_size=0.2 , random_state=42)
X_train , X_val , y_train , y_val = train_test_split(X_train ,y_train , test_size=0.1 , random_state=42)




print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

In [ ]:
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
base_model = MobileNetV2(
    input_shape = (96,96,3),
    include_top = False,
    weights = 'imagenet'
)

base_model.trainable = False

model = keras.Sequential([
    base_model,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(64 , activation = 'relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(4 , activation = 'softmax')
])

In [ ]:
model.compile(
    optimizer = 'adam',
    loss = 'sparse_categorical_crossentropy',
    metrics =['accuracy']
)

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,         
    restore_best_weights=True,
    mode = 'min'
)

model.fit(X_train, y_train,
          epochs=20,
          batch_size=32,
          validation_data=(X_val, y_val),
          callbacks=[early_stop])

In [ ]:
# Test accuracy
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.2%}")